# Restaurant Bot: Guardrails & Complaints

OpenAI Agents SDK + Clean Architecture 기반 레스토랑 봇

## Architecture
```
domain/          - 엔티티 (MenuItem, Reservation, Order, Complaint)와 리포지토리 포트
application/     - Use Cases, Context (유스케이스 조율)
infrastructure/  - 인메모리 리포지토리 구현, RunConfig
presentation/    - Agent/Tools/Guardrails/Models (프레젠테이션)
```

## 구성 요소
- **Triage Agent** (입력 가드레일 장착) → 전문 에이전트로 라우팅
- **Menu / Reservation / Order Agents** — 각 도메인 전담
- **Complaints Agent** (출력 가드레일 장착) — 불만 공감/사과/해결책 제시
- **Input Guardrail** — off-topic, 부적절한 언어 차단
- **Output Guardrail** — 비전문적/내부 정보 노출 응답 차단

## 1. 환경 설정

In [16]:
import sys, os, pathlib

# VS Code Jupyter는 __vsc_ipynb_file__ 로 노트북 절대 경로를 제공함
_nb_path = globals().get("__vsc_ipynb_file__") or os.path.abspath("main.ipynb")
_nb_dir = str(pathlib.Path(_nb_path).parent.resolve())

if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)
os.chdir(_nb_dir)
print(f"경로 설정 완료: {_nb_dir}")

경로 설정 완료: /Users/hans/Desktop/project/movie-expert-agent/restaurant-bot


In [17]:
from infrastructure.config import load_run_config

run_config = load_run_config()
print("환경 설정 완료")

환경 설정 완료


## 2. DI Composition Root - 의존성 조립

In [ ]:
from infrastructure.memory_repositories import (
    InMemoryMenuRepository,
    InMemoryOrderRepository,
    InMemoryReservationRepository,
)
from application.context import AppContext
from application.use_cases import (
    GetMenuUseCase,
    MakeReservationUseCase,
    PlaceOrderUseCase,
)

menu_repo = InMemoryMenuRepository()
reservation_repo = InMemoryReservationRepository()
order_repo = InMemoryOrderRepository()

context = AppContext(
    get_menu=GetMenuUseCase(menu_repo),
    make_reservation=MakeReservationUseCase(reservation_repo),
    place_order=PlaceOrderUseCase(menu_repo, order_repo),
)
print("의존성 조립 완료")

## 3. Agent 생성

In [19]:
from presentation.agents import create_triage_agent

triage_agent = create_triage_agent()
current_agent = triage_agent

print(f"Agent: {triage_agent.name}")
print(f"Handoffs: {[h.agent_name for h in triage_agent.handoffs]}")
print(f"Input Guardrails: {len(triage_agent.input_guardrails)}")

Agent: Triage_Agent
Handoffs: ['Menu_Agent', 'Reservation_Agent', 'Order_Agent', 'Complaints_Agent']
Input Guardrails: 1


## 4. 대화 실행

아래 셀을 실행하면 대화를 시작합니다. 빈 입력으로 엔터를 치면 종료됩니다.  
매 턴마다 Triage Agent가 최신 메시지를 보고 적절한 전문 에이전트로 라우팅합니다.

In [20]:
# 대화 히스토리 초기화
input_items = []

In [ ]:
from agents import Runner, InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered

print("대화를 시작합니다. 빈 입력으로 엔터를 치면 종료됩니다.\n")

while True:
    user_input = input("고객: ").strip()
    if not user_input:
        print("대화를 종료합니다.")
        break

    input_items.append({"role": "user", "content": user_input})
    try:
        result = await Runner.run(triage_agent, input_items, context=context, run_config=run_config)
        input_items = result.to_input_list()
        current_agent = result.last_agent
        print(f"\n[{current_agent.name}] {result.final_output}\n")
    except InputGuardrailTripwireTriggered as e:
        info = e.guardrail_result.output.output_info
        input_items.pop()
        print(f"\n[Guardrail 차단] {info.reason}\n")
    except OutputGuardrailTripwireTriggered:
        print("\n[Output Guardrail 차단] 응답이 가드레일에 의해 차단되었습니다.\n")